# 07 — Expected Returns: Building Block Decomposition

## Overview

This notebook documents how expected returns are calculated for the KSE 100 Portfolio Builder Phase 2.

Instead of hardcoding 22%, 15%, and 5% as in Phase 1, Phase 2 uses the **building block identity**:

> **Expected Return = Dividend Yield + Real Earnings Growth + Inflation + Valuation Change**

Observed values (dividend yield, current P/E) are read dynamically from `data/processed/macro_snapshot.csv`. Forecast assumptions (real earnings growth, ending P/E) come from `config/scenarios.yaml`.

When macro data changes, expected returns recalculate automatically.

## 1. Load the Building Block Module

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from kse.blocks import (
    get_macro_values,
    expected_return,
    calculate_all_expected_returns,
    print_building_block_table,
    valuation_change_annualized
)
from kse.scenarios import print_scenario_table, get_all_scenarios

## 2. Current Macro Values

In [ ]:
macro = get_macro_values()
print("Current Macro Values (from macro_snapshot.csv):")
print(f"  Dividend Yield: {macro['dividend_yield']:.1%}")
print(f"  KSE 100 P/E:   {macro['kse100_pe']:.1f}x")
print()
print("These values are fetched dynamically from SBP, PBS, and PSX sources.")
print("They are NOT hardcoded in Python code.")

## 3. Building Block Decomposition

### The Identity

For Bull and Base scenarios, the standard identity works:
- Expected Return = Dividend Yield + Real Earnings Growth + Inflation + Valuation Change

For the Bear scenario, nominal earnings growth is used directly because the identity breaks down in crisis (nominal earnings lag inflation due to margin compression, currency devaluation, and import compression).

### Valuation Change

The valuation term is the annualized change in P/E over the horizon:
- Bull: P/E re-rates from ~7x to 11x → +4.6%/year
- Base: P/E stays flat → 0%
- Bear: P/E de-rates from ~7x to 3.8x → -6.0%/year

In [ ]:
# Calculate valuation change for each scenario
print("Valuation Change Calculation:")
print("=" * 60)
for scenario in ["Bull", "Base", "Bear"]:
    ret, val_change, components = expected_return(scenario)
    pe_start = components['pe_start']
    pe_end = components['pe_end']
    horizon = components['horizon_years']
    
    if pe_end == "flat":
        print(f"  {scenario}: P/E stays flat at {pe_start:.1f}x → 0.0%/year")
    else:
        print(f"  {scenario}: P/E from {pe_start:.1f}x to {pe_end:.1f}x over {horizon} years → {val_change:+.1%}/year")
print("=" * 60)

In [ ]:
# Print the full building block table
results = print_building_block_table()

## 4. Full Scenario Table

The income sleeve return is calculated from the policy rate:
- Income Return = Policy Rate - 1% (approximating money market fund returns)

This captures the inverse relationship: the income sleeve earns MORE in the Bear case (16.5%) because rates spike, hedging the equity crash.

In [ ]:
print_scenario_table()

## 5. Sensitivity Analysis

In [ ]:
# How does the Base case change if inflation is 10% instead of 7%?
print("Sensitivity: Base case with different inflation assumptions")
print("=" * 60)

from kse.blocks import BUILDING_BLOCKS, SCENARIOS_CFG

base_inflation = SCENARIOS_CFG["Base"]["inflation"]
macro = get_macro_values()

for infl in [0.05, 0.07, 0.10, 0.13]:
    # Override inflation
    div_y = macro["dividend_yield"]
    real_g = BUILDING_BLOCKS["Base"]["real_earnings_growth"]
    val_c = valuation_change_annualized(macro["kse100_pe"], "flat", 10)
    ret = div_y + real_g + infl + val_c
    print(f"  Inflation {infl:.0%}: Expected return = {ret:.1%}")

print("=" * 60)
print("This shows how the model responds to changing macro conditions.")

## 6. Validation

The building block returns should fall within the scenario ranges defined in the proposals doc:
- Bull: 20-24% → calculated: ~20%
- Base: 14-17% → calculated: ~16.5%
- Bear: 0-6% → calculated: ~4.5%

The differences are within the scenario ranges and are the kind of thing the calibration test surfaces.

In [ ]:
# Validate that building block returns are reasonable
results = calculate_all_expected_returns()
print("Validation:")
print("=" * 60)
ranges = {"Bull": (0.20, 0.24), "Base": (0.14, 0.17), "Bear": (0.00, 0.06)}
for scenario in ["Bull", "Base", "Bear"]:
    ret = results[scenario]["expected_return"]
    lo, hi = ranges[scenario]
    status = "✓" if lo <= ret <= hi else "⚠"
    print(f"  {scenario}: {ret:.1%} (range: {lo:.0%}-{hi:.0%}) {status}")
print("=" * 60)